In [1]:
# ── View aligned outputs in napari ─────────────────────────────────────────────
import napari
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from scipy.ndimage import maximum_filter, distance_transform_edt as edt_func
import numpy as np
import sknw
from skimage.morphology import skeletonize as skeletonize_3d
from scipy.ndimage import distance_transform_edt as edt
import sys
sys.path.insert(0, str(Path(r"C:\Users\taylorhearn\git_repos\vascumap\bel_vascumap")))

from skeletonisation import (prune_graph, graph2image, compute_cross_sectional_areas,
                              remove_mid_node, collect_border_vicinity_edges)


In [2]:
# ── Trim top/bottom slices where >80% of the slice is segmented ───────────────
# Only peels from the outer edges — middle slices are never removed.


def trim_segmentation(segmentation, fill_threshold = 0.75):
    slice_fill = segmentation.astype(bool).mean(axis=(1, 2))  # (Z,) fraction per slice
    keep_start = 0
    while keep_start < len(slice_fill) and slice_fill[keep_start] > fill_threshold:
        keep_start += 1
    keep_end = len(slice_fill) - 1
    while keep_end >= keep_start and slice_fill[keep_end] > fill_threshold:
        keep_end -= 1

    return segmentation [keep_start:keep_end+1]


In [3]:
def make_overlay(seg_bg, arr, cmap='cool'):
    thick = maximum_filter(np.sum(arr.astype(np.float32), axis=0), size=3)
    norm  = thick / max(thick.max(), 1)
    rgb   = np.stack([seg_bg * 0.40] * 3, axis=-1)
    mask  = norm > 0
    rgb[mask] = cm.get_cmap(cmap)(norm[mask])[:, :3]
    return np.clip(rgb, 0, 1)

def draw_graph(ax, g, diams, title, background, sm):
    ax.imshow(background)
    for (u, v), diam in zip(g.edges(), diams):
        try:
            pts = g[u][v]['pts'].astype(int)
            ax.plot(pts[:,2], pts[:,1], color=sm.to_rgba(diam), linewidth=2.0, solid_capstyle='round')
        except (KeyError, IndexError):
            continue
    nx_x, nx_y, nc = [], [], []
    for node in g.nodes():
        pos = g.nodes[node]['pts']
        if pos.ndim > 1:
            pos = pos[0]
        nx_x.append(pos[2]); nx_y.append(pos[1])
        nc.append('lime' if g.degree(node) == 1 else 'white')
    if nx_x:
        ax.scatter(nx_x, nx_y, c=nc, s=20, alpha=0.8, zorder=5)
    ax.set_title(f'{title}\n({g.number_of_nodes()} nodes, {g.number_of_edges()} edges)', fontsize=13)
    ax.set_aspect('equal', adjustable='box')
        
        
def skeletonise_and_graph(segmentation, view_dir, voxel_size_um=(2.0,2.0,2.0), save_path=None):
    skeleton   = skeletonize_3d(segmentation.astype(bool)).astype(np.uint8)

    binary_edt = edt(segmentation.astype(bool))
    graph      = sknw.build_sknw(skeleton.astype(bool))

    area_image = compute_cross_sectional_areas(
        segmentation, skeleton, binary_edt, voxel_size_um=voxel_size_um,
    )

    for n in list(graph.nodes()):
        graph.nodes[n]['pts']    = graph.nodes[n]['pts'][0]
        graph.nodes[n]['sprout'] = graph.degree(n) == 1

    pruned_graph = prune_graph(graph=graph, area_3d=area_image, edt_cutoff=0.20, length_cutoff=25)
    clean_graph  = remove_mid_node(pruned_graph)
    clean_graph  = collect_border_vicinity_edges(clean_graph, segmentation.shape)
    isolated     = [n for n in clean_graph.nodes() if clean_graph.degree[n] == 0]
    if isolated:
        clean_graph.remove_nodes_from(isolated)
    for n in clean_graph.nodes():
        clean_graph.nodes[n]['sprout'] = clean_graph.degree(n) == 1

    clean_skeleton = graph2image(clean_graph, segmentation.shape).astype(np.uint8)


    nz         = segmentation.shape[0]
    seg_bool   = segmentation.astype(bool)
    seg_max    = np.mean(seg_bool, axis=0).astype(np.float32)
    background = np.stack([seg_max * 0.40] * 3, axis=-1)
    binary_edt = edt_func(seg_bool)

    overlay_skel  = make_overlay(seg_max, skeleton)
    overlay_clean = make_overlay(seg_max, clean_skeleton)

    # ── Graph overlays — both built from clean_skeleton ──────────────────────────
    raw_graph = sknw.build_sknw(clean_skeleton.astype(bool))
    for _n in list(raw_graph.nodes()):
        if raw_graph.nodes[_n]['pts'].ndim > 1:
            raw_graph.nodes[_n]['pts'] = raw_graph.nodes[_n]['pts'][0]

    def edge_diameters_for(g):
        diams = []
        for u, v in g.edges():
            try:
                pts   = np.clip(g[u][v]['pts'].astype(int), [0,0,0], [s-1 for s in binary_edt.shape])
                radii = binary_edt[pts[:,0], pts[:,1], pts[:,2]]
                diams.append(float(np.median(radii)) * 2.0)
            except (KeyError, IndexError):
                diams.append(0.0)
        return diams

    raw_diams   = edge_diameters_for(raw_graph)
    clean_diams = edge_diameters_for(clean_graph)

    all_diams = [d for d in raw_diams + clean_diams if d > 0]
    vmin   = np.percentile(all_diams, 5) * 0.5 if all_diams else 0
    vmax   = np.percentile(all_diams, 95)      if all_diams else 1
    norm_g = Normalize(vmin=vmin, vmax=vmax)
    sm     = cm.ScalarMappable(norm=norm_g, cmap=cm.magma)


        

    # ── Figure ────────────────────────────────────────────────────────────────────
    plt.style.use('dark_background')
    fig, ax = plt.subplots(ncols=4, figsize=(24, 16))

    ax[0].imshow(overlay_skel)
    ax[0].set_title(f'Skeleton  ({skeleton.sum():,} voxels,  {nz} z-slices)', fontsize=13)

    ax[1].imshow(overlay_clean)
    ax[1].set_title(f'Clean skeleton  ({clean_skeleton.sum():,} voxels,  {nz} z-slices)', fontsize=13)


    draw_graph(ax[2], raw_graph,   raw_diams,   'Graph (clean skeleton)', background, sm)
    draw_graph(ax[3], clean_graph, clean_diams, 'Clean graph (pruned)', background, sm)

    fig.colorbar(sm, ax=ax[-1], fraction=0.02, pad=0.02, label='Vessel diameter (µm)')
    for a in ax:
        a.axis("off")
    plt.suptitle(view_dir.name, fontsize=16)
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()




In [ ]:
# ── Batch processing: skeletonise all Vascumap outputs ────────────────────────
from pathlib import Path

root_dir  = Path(r"Z:\Bel\Vascumap_Outputs")
out_dir   = root_dir / "Graphs"
out_dir.mkdir(exist_ok=True)

subfolders = sorted([d for d in root_dir.iterdir() if d.is_dir() and d.name != "Graphs"])
print(f"Found {len(subfolders)} subfolders")

for folder in subfolders:
    seg_files = sorted(folder.glob("*_clean_segmentation.npy"))
    if not seg_files:
        print(f"  SKIP  {folder.name}  (no clean_segmentation file)")
        continue

    seg_path = seg_files[0]
    save_path = out_dir / f"{folder.name}.png"
    if save_path.exists():
        print(f"  SKIP  {folder.name}  (already done)")
        continue

    print(f"  Processing  {folder.name} …")
    try:
        seg = np.load(str(seg_path))
        seg = trim_segmentation(seg)
        skeletonise_and_graph(seg, folder, save_path=save_path)
        print(f"    → saved {save_path.name}")
    except Exception as e:
        print(f"    ERROR: {e}")

print("Done.")


Found 58 subfolders
  Processing  20250619_Vascumap_Dev25_11_Daisy10_20250619_Vascumap_Dev25_11_Daisy10 …


C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_697036\3577462839.py:6: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  rgb[mask] = cm.get_cmap(cmap)(norm[mask])[:, :3]


    → saved 20250619_Vascumap_Dev25_11_Daisy10_20250619_Vascumap_Dev25_11_Daisy10.png
  Processing  20250619_Vascumap_Dev25_11_Wicell10_2_20250619_Vascumap_Dev25_11_Wicell10_2 …


C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_697036\3577462839.py:6: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  rgb[mask] = cm.get_cmap(cmap)(norm[mask])[:, :3]


    → saved 20250619_Vascumap_Dev25_11_Wicell10_2_20250619_Vascumap_Dev25_11_Wicell10_2.png
  Processing  20250619_Vascumap_Dev25_11_WicellRescue_20250619_Vascumap_Dev25_11_WicellRescue …


...

    → saved Akinola_09102024_day7_hmvec_cs_merged_img2_vessels_only_v4_Merged.png
  Processing  Akinola_09102024_day7_hmvec_cs_merged_img3_CS_vessels_n1_Merged …


C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_697036\3577462839.py:6: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  rgb[mask] = cm.get_cmap(cmap)(norm[mask])[:, :3]


    → saved Akinola_09102024_day7_hmvec_cs_merged_img3_CS_vessels_n1_Merged.png
  Processing  Akinola_09102024_day7_hmvec_cs_merged_img4_CS_vessels_n2_Merged …


C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_697036\3577462839.py:6: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  rgb[mask] = cm.get_cmap(cmap)(norm[mask])[:, :3]


    → saved Akinola_09102024_day7_hmvec_cs_merged_img4_CS_vessels_n2_Merged.png
  Processing  Akinola_09102024_day7_hmvec_cs_merged_img5_CS_vessels_n3_Merged …


In [ ]:
# ── VMTK centreline skeletonisation ───────────────────────────────────────────
# Install: conda install -c conda-forge vmtk vtk
#
# Seed points are detected automatically: each connected vessel component that
# touches any image face contributes one representative point (its centroid).
# The first such point is the source; the rest are targets — no user input needed.

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.ndimage import gaussian_filter, maximum_filter, generate_binary_structure, label as ndi_label

try:
    import vtk
    from vmtk import vmtkscripts
    from skimage.measure import marching_cubes
    HAS_VMTK = True
except ImportError:
    HAS_VMTK = False
    print("VMTK/VTK not found. Install with:\n  conda install -c conda-forge vmtk vtk")

def _border_touching_seeds_um(mask, spacing):
    """Return seed points in physical µm (XYZ for VTK) for each connected
    component that touches any face of the volume."""
    structure = generate_binary_structure(3, 2)
    labels, nlab = ndi_label(mask.astype(bool), structure=structure)

    border = np.zeros_like(mask, dtype=bool)
    border[0, :, :] = border[-1, :, :] = True
    border[:, 0, :] = border[:, -1, :] = True
    border[:, :, 0] = border[:, :, -1] = True

    touching = np.unique(labels[border & (labels > 0)])
    seeds = []
    for lab in touching:
        zyx = np.argwhere(labels == lab)
        c = zyx.mean(axis=0)               # centroid in voxel ZYX
        # convert to physical XYZ (swap axis order, multiply by spacing)
        seeds.append((float(c[2] * spacing[2]),
                      float(c[1] * spacing[1]),
                      float(c[0] * spacing[0])))
    return seeds

if HAS_VMTK:
    SPACING = (2.0, 2.0, 2.0)   # µm per voxel: (Z, Y, X)

    # 1. Light Gaussian smooth to reduce surface noise
    seg_f = gaussian_filter(segmentation.astype(np.float32), sigma=0.8)

    # 2. Marching-cubes surface in physical µm space
    verts, faces, _, _ = marching_cubes(seg_f, level=0.5, spacing=SPACING)

    # 3. Build VTK PolyData (VTK uses XYZ → swap from ZYX)
    vtk_pts = vtk.vtkPoints()
    for v in verts:
        vtk_pts.InsertNextPoint(float(v[2]), float(v[1]), float(v[0]))
    vtk_tris = vtk.vtkCellArray()
    for tri in faces:
        vtk_tris.InsertNextCell(3, tri.tolist())
    surface_pd = vtk.vtkPolyData()
    surface_pd.SetPoints(vtk_pts)
    surface_pd.SetPolys(vtk_tris)

    # 4. Cap the surface
    capper = vmtkscripts.vmtkSurfaceCapper()
    capper.Surface = surface_pd
    capper.Execute()

    # 5. Auto-detect seed points from border-touching vessel components
    seeds = _border_touching_seeds_um(segmentation.astype(bool), SPACING)
    if len(seeds) < 2:
        raise RuntimeError(
            f"Only {len(seeds)} border-touching component(s) found. "
            "Need at least 2 (source + target). Check the segmentation."
        )
    print(f"Auto-detected {len(seeds)} seed point(s) from border-touching components")

    source_pts  = list(seeds[0])          # flat XYZ of first seed
    target_pts  = [c for s in seeds[1:] for c in s]  # flat XYZ of remaining seeds

    # 6. Extract centrelines using programmatic seed points
    cl = vmtkscripts.vmtkCenterlines()
    cl.Surface = capper.Surface
    cl.SeedSelectorName = "pointlist"
    cl.SourcePoints = source_pts
    cl.TargetPoints = target_pts
    cl.AppendEndPoints = 1
    cl.Execute()

    # 7. Map centreline points back to voxel coordinates (VTK XYZ µm → ZYX idx)
    npts = cl.Centerlines.GetNumberOfPoints()
    vmtk_skel = np.zeros_like(segmentation, dtype=np.uint8)
    for i in range(npts):
        xm, ym, zm = cl.Centerlines.GetPoint(i)
        zi = int(round(zm / SPACING[0]))
        yi = int(round(ym / SPACING[1]))
        xi = int(round(xm / SPACING[2]))
        vmtk_skel[
            np.clip(zi, 0, vmtk_skel.shape[0] - 1),
            np.clip(yi, 0, vmtk_skel.shape[1] - 1),
            np.clip(xi, 0, vmtk_skel.shape[2] - 1),
        ] = 1

    # 8. Visualise
    nz = segmentation.shape[0]
    skel_thick = maximum_filter(np.sum(vmtk_skel.astype(np.float32), axis=0), size=3)
    skel_norm  = skel_thick / max(skel_thick.max(), 1)
    seg_max    = np.max(segmentation.astype(bool), axis=0).astype(np.float32)
    cmap_fn    = cm.get_cmap('cool')
    rgb = np.stack([seg_max * 0.15] * 3, axis=-1)
    mask = skel_norm > 0
    rgb[mask] = cmap_fn(skel_norm[mask])[:, :3]

    plt.style.use('dark_background')
    fig, ax = plt.subplots(figsize=(12, 12))
    ax.imshow(np.clip(rgb, 0, 1))
    ax.set_title(f'VMTK centrelines sum-proj  ({npts:,} pts,  {nz} z-slices)', fontsize=14)
    ax.axis('off')
    plt.suptitle(view_dir.name, fontsize=16)
    plt.tight_layout()
    plt.show()


Please input boundary ids: 

In [2]:
# ── kimimaro: TEASAR skeletonisation ──────────────────────────────────────────
# kimimaro implements TEASAR: topologically-correct skeletonisation designed
# specifically for tube-like structures.
# Install: pip install kimimaro
#
# Key parameters to tune:
#   scale      — penalty scale (larger = fewer, longer branches kept)
#   const      — constant penalty offset in voxels (prunes short stubs)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.ndimage import maximum_filter

try:
    import kimimaro
    HAS_KIMIMARO = True
except ImportError:
    HAS_KIMIMARO = False
    print("kimimaro not found. Install with:  pip install kimimaro")

if HAS_KIMIMARO:
    SPACING = (2.0, 2.0, 2.0)   # µm per voxel (Z, Y, X)

    skels = kimimaro.skeletonize(
        segmentation.astype(np.uint8),
        teasar_params={
            'scale': 1.5,
            'const': 300,
            'pdrf_exponent': 4,
            'pdrf_scale': 100000,
            'soma_detection_threshold': 1100,
            'soma_acceptance_threshold': 3500,
            'soma_invalidation_scale': 1.0,
            'soma_invalidation_const': 300,
            'max_paths': None,
        },
        anisotropy=SPACING,
        dust_threshold=500,
        progress=True,
    )

    # Accumulate all skeleton vertices into a binary volume
    # kimimaro vertices are in physical space (µm in ZYX order)
    kimimaro_skel = np.zeros_like(segmentation, dtype=np.uint8)
    total_pts = 0
    for label, skel in skels.items():
        for v in skel.vertices:
            zi = int(round(v[0] / SPACING[0]))
            yi = int(round(v[1] / SPACING[1]))
            xi = int(round(v[2] / SPACING[2]))
            kimimaro_skel[
                np.clip(zi, 0, kimimaro_skel.shape[0] - 1),
                np.clip(yi, 0, kimimaro_skel.shape[1] - 1),
                np.clip(xi, 0, kimimaro_skel.shape[2] - 1),
            ] = 1
            total_pts += 1
    print(f"kimimaro: {len(skels)} components, {total_pts:,} skeleton vertices")

    nz = segmentation.shape[0]
    skel_thick = maximum_filter(np.sum(kimimaro_skel.astype(np.float32), axis=0), size=3)
    skel_norm  = skel_thick / max(skel_thick.max(), 1)
    seg_max    = np.max(segmentation.astype(bool), axis=0).astype(np.float32)
    cmap_fn    = cm.get_cmap('cool')
    rgb = np.stack([seg_max * 0.15] * 3, axis=-1)
    mask = skel_norm > 0
    rgb[mask] = cmap_fn(skel_norm[mask])[:, :3]

    plt.style.use('dark_background')
    fig, ax = plt.subplots(figsize=(12, 12))
    ax.imshow(np.clip(rgb, 0, 1))
    ax.set_title(f'kimimaro TEASAR sum-proj  ({total_pts:,} pts,  {nz} z-slices)', fontsize=14)
    ax.axis('off')
    plt.suptitle(view_dir.name, fontsize=16)
    plt.tight_layout()
    plt.show()


NameError: name 'segmentation' is not defined

In [4]:
# ── pc-skeletor: Laplacian-based contraction (LBC) ────────────────────────────
# pc-skeletor is already in the workspace at pc-skeletor/
# Install open3d with:  pip install open3d
#
# LBC iteratively contracts a point cloud toward its medial axis.
# Input: surface voxels of the segmentation (foreground minus interior),
# subsampled to ≤30k points to keep it tractable.

import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.ndimage import maximum_filter, binary_erosion


try:
    import open3d as o3d
    from pc_skeletor import LBC
    HAS_LBC = True
except ImportError:
    HAS_LBC = False
    print("pc-skeletor or open3d not found. Install with:  pip install open3d")

if HAS_LBC:
    SPACING = np.array([2.0, 2.0, 2.0])   # µm per voxel (Z, Y, X)

    # Surface voxels = foreground minus eroded interior
    seg_bool = segmentation.astype(bool)
    surface  = seg_bool & ~binary_erosion(seg_bool)
    zyx_idx  = np.argwhere(surface)
    pts_um   = zyx_idx * SPACING             # (N, 3) ZYX in µm

    MAX_PTS = 30_000
    if len(pts_um) > MAX_PTS:
        rng    = np.random.default_rng(0)
        pts_um = pts_um[rng.choice(len(pts_um), MAX_PTS, replace=False)]
    print(f"LBC input: {len(pts_um):,} surface points")

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts_um[:, ::-1])  # ZYX → XYZ for open3d

    lbc = LBC(point_cloud=pcd,
              down_sample=4.0,
              init_contraction=2,
              init_attraction=0.5,
              max_iteration_steps=15)
    lbc.extract_skeleton()
    lbc.extract_topology()

    # Skeleton points: open3d gives XYZ → flip to ZYX → back to voxel indices
    skel_pts_xyz = np.asarray(lbc.skeleton.points)
    skel_pts_zyx = skel_pts_xyz[:, ::-1]
    lbc_skel = np.zeros_like(segmentation, dtype=np.uint8)
    for pt in skel_pts_zyx:
        zi = int(round(pt[0] / SPACING[0]))
        yi = int(round(pt[1] / SPACING[1]))
        xi = int(round(pt[2] / SPACING[2]))
        lbc_skel[
            np.clip(zi, 0, lbc_skel.shape[0] - 1),
            np.clip(yi, 0, lbc_skel.shape[1] - 1),
            np.clip(xi, 0, lbc_skel.shape[2] - 1),
        ] = 1

    nz = segmentation.shape[0]
    skel_thick = maximum_filter(np.sum(lbc_skel.astype(np.float32), axis=0), size=3)
    skel_norm  = skel_thick / max(skel_thick.max(), 1)
    seg_max    = np.max(seg_bool, axis=0).astype(np.float32)
    cmap_fn    = cm.get_cmap('cool')
    rgb = np.stack([seg_max * 0.15] * 3, axis=-1)
    mask = skel_norm > 0
    rgb[mask] = cmap_fn(skel_norm[mask])[:, :3]

    plt.style.use('dark_background')
    fig, ax = plt.subplots(figsize=(12, 12))
    ax.imshow(np.clip(rgb, 0, 1))
    ax.set_title(f'pc-skeletor LBC sum-proj  ({len(skel_pts_xyz):,} pts,  {nz} z-slices)', fontsize=14)
    ax.axis('off')
    plt.suptitle(view_dir.name, fontsize=16)
    plt.tight_layout()
    plt.show()

pc-skeletor or open3d not found. Install with:  pip install open3d


In [ ]:
# ── DGtal: isthmus-based thinning (VoxelComplex) ──────────────────────────────
# Install: pip install dgtal
# Implements Couprie & Bertrand's asymmetric parallel 3D thinning.
# skel_type='isthmus' preserves 1-isthmus voxels (bridges/tube crossings),
# giving a topologically correct, centred skeleton for vascular networks.
# select_type='dmax' picks the voxel furthest from the surface → centred result.

import numpy as np
from skimage.measure import block_reduce

mask = segmentation.astype(np.uint8)   # shape (Z, Y, X), values 0/1

# Downsample by 4 in each dimension using block mean
mask_ds_prob = block_reduce(mask, block_size=(8, 8, 8), func=np.mean)

# Re-binarise
mask_ds = mask_ds_prob >= 0.25   # try 0.25, 0.5, 0.75


import dgtal
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.ndimage import maximum_filter

Point  = dgtal.kernel.Point3D
Domain = dgtal.kernel.DomainZ3i
DSet   = dgtal.kernel.DigitalSetZ3i
KSpace = dgtal.topology.KSpace3D
VC     = dgtal.topology.VoxelComplex
CData  = dgtal.topology.CubicalCellData

SPACING     = (2.0, 2.0, 2.0)  # µm per voxel (Z, Y, X)
# ── Tunable ────────────────────────────────────────────────────────────────────
SKEL_TYPE   = 'isthmus'   # 'end' | 'ulti' | 'isthmus'
SELECT_TYPE = 'dmax'      # 'first' | 'random' | 'dmax'
PERSISTENCE = 0           # >0 prunes branches shorter than this (voxels)
# ───────────────────────────────────────────────────────────────────────────────

nz, ny, nx = mask_ds.shape

# 1. Khalimsky space
lower = Point(0, 0, 0)
upper = Point(nx - 1, ny - 1, nz - 1)   # DGtal uses XYZ ordering
ks = KSpace()
ks.init(lower, upper, True)

# 2. Build a DigitalSetZ3i from the foreground voxels
#    numpy indices are ZYX; DGtal Point3D expects XYZ
domain = Domain(lower, upper)
dset   = DSet(domain)
fg_idx = np.argwhere(mask_ds)
print(f"Inserting {len(fg_idx):,} foreground voxels into DGtal set …")
for z, y, x in fg_idx:
    dset.insert(Point(int(x), int(y), int(z)))

# 3. VoxelComplex → thinning
vc = VC(ks)
vc.construct(dset)
print("Running DGtal thinning …")
skel_vc = vc.thinningVoxelComplex(
    SKEL_TYPE, SELECT_TYPE, str(dgtal.tables_folder),
    persistence=PERSISTENCE, verbose=True,
)

# 4. Dump result back to a DigitalSet, then to a numpy volume
dump_set   = DSet(domain)
skel_vc.dumpVoxels(dump_set)
dgtal_skel = np.zeros_like(mask_ds, dtype=np.uint8)
for pt in dump_set:
    xi, yi, zi = pt[0], pt[1], pt[2]   # DGtal XYZ → numpy ZYX
    dgtal_skel[
        np.clip(zi, 0, nz - 1),
        np.clip(yi, 0, ny - 1),
        np.clip(xi, 0, nx - 1),
    ] = 1
print(f"DGtal skeleton voxels: {dgtal_skel.sum():,}")

# 5. Visualise
skel_thick = maximum_filter(np.sum(dgtal_skel.astype(np.float32), axis=0), size=3)
skel_norm  = skel_thick / max(skel_thick.max(), 1)
seg_max    = np.max(mask_ds.astype(bool), axis=0).astype(np.float32)
cmap_fn    = cm.get_cmap('cool')
rgb        = np.stack([seg_max * 0.15] * 3, axis=-1)
mask       = skel_norm > 0
rgb[mask]  = cmap_fn(skel_norm[mask])[:, :3]

plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(np.clip(rgb, 0, 1))
ax.set_title(
    f'DGtal isthmus thinning sum-proj  ({dgtal_skel.sum():,} voxels,  {nz} z-slices)',
    fontsize=14,
)
ax.axis('off')
plt.suptitle(view_dir.name, fontsize=16)
plt.tight_layout()
plt.show()


Inserting 1,227,802 foreground voxels into DGtal set …
